In [51]:
import pandas
from uuid import uuid4
import random
from datetime import date
from dateutil.relativedelta import relativedelta

In [52]:
# maps 
territory_map = {
    "t1": "North",
    "t2": "South",
    "t3": "West",
    "t4": "East"
}

seller_map = {
    "u1": "John Doe",
    "u2": "Jane Doe",
    "u3": "Bill Tack",
    "u4": "Peter Martin",
    "u5": "Michael Block",
    "u6": "Denise Waters",
    "u7": "Anna White",
    "u8": "Mia Sears",
    "u9": "Daniel Metters",
    "u0": "Lina Sanders",
}

In [53]:
n_samples_transactions = 1000

uuids = [str(uuid4()) for _ in range(n_samples_transactions)]

territories = random.choices(list(territory_map.keys()), k=n_samples_transactions)
sellers = random.choices(list(seller_map.keys()), k=n_samples_transactions)
amounts = [round(random.random()*10_000, 2) for _ in range(n_samples_transactions)]

# compute random days between Jan 1st, 2023 and June 30th, 2026
# by creating random offsets from the initial date, bounded to the end date
date_ini = date(2023, 1, 1)
date_end = date(2026, 6, 30)
date_diff_days = (date_end - date_ini).days
index_dates = range(0, date_diff_days+1)
days_offset = random.choices(index_dates, k=n_samples_transactions)

# some days (2/7) are weekends, so to make it more realistic
# we pick a random set with replacement from those days we know are week days
# and paste them together
random_dates = [date_ini + relativedelta(days=day_offset) for day_offset in days_offset]
week_days = [d for d in random_dates if d.isoweekday() < 6]
transaction_dates = week_days + random.choices(week_days, k=n_samples_transactions - len(week_days))

In [54]:
from IPython.display import display, HTML
import random as _random
_rng = _random.Random(42)
_seller_territories = [_rng.choice(list(territory_map.keys())) for _ in seller_map]
transaction_df = pandas.DataFrame(
    {
        "transaction_id": uuids,
        "transaction_date": transaction_dates,
        "seller_id": sellers,
        "amount_eur": amounts
    }
)
seller_df = pandas.DataFrame({
    "seller_id": seller_map.keys(),
    "territory_id": _seller_territories,
    "seller_name": seller_map.values(),
})

territory_df = pandas.DataFrame({
    "territory_id": territory_map.keys(),
    "territory_name": territory_map.values(),
})

display(HTML(transaction_df.head().to_html(index=False)))
display(HTML(seller_df.head().to_html(index=False)))
display(HTML(territory_df.head().to_html(index=False)))

transaction_id,transaction_date,seller_id,amount_eur
11c54a2e-6036-42e8-bfc0-3624ac65a74c,2025-06-11,u8,2592.92
a1476ab3-86cf-45e4-8206-c3ecf07baeaf,2026-04-10,u3,2683.38
2eca9762-34db-4852-9af5-78e049050851,2025-01-22,u1,8864.86
49d76709-c4b6-4778-971a-3916a4febd09,2023-11-14,u3,5826.42
629f394e-092e-4994-aaa6-2f295f8d7be0,2024-12-12,u9,9969.75


seller_id,territory_id,seller_name
u1,t1,John Doe
u2,t1,Jane Doe
u3,t3,Bill Tack
u4,t2,Peter Martin
u5,t2,Michael Block


territory_id,territory_name
t1,North
t2,South
t3,West
t4,East


In [55]:
transaction_df.to_csv("./transactions.csv", index=False)
seller_df.to_csv("./seller.csv", index=False)
territory_df.to_csv("./territory.csv", index=False)

In [56]:
transactions = transaction_df.copy()

In [57]:
transactions["transaction_date"] = pandas.to_datetime(transactions["transaction_date"])
transactions["transaction_month"] = transactions["transaction_date"].transform(lambda x: x.replace(day=1))

In [61]:
transactions_with_territory = transactions.merge(
    seller_df,
    on=["seller_id"]
).merge(
    territory_df,
    on=["territory_id"]
)

transactions_with_territory.head()

,transaction_id,transaction_date,seller_id,amount_eur,transaction_month,territory_id,seller_name,territory_name
0,11c54a2e-6036-42e8-bfc0-3624ac65a74c,2025-06-11,u8,2592.92,2025-06-01,t1,Mia Sears,North
1,a1476ab3-86cf-45e4-8206-c3ecf07baeaf,2026-04-10,u3,2683.38,2026-04-01,t3,Bill Tack,West
2,2eca9762-34db-4852-9af5-78e049050851,2025-01-22,u1,8864.86,2025-01-01,t1,John Doe,North
3,49d76709-c4b6-4778-971a-3916a4febd09,2023-11-14,u3,5826.42,2023-11-01,t3,Bill Tack,West
4,629f394e-092e-4994-aaa6-2f295f8d7be0,2024-12-12,u9,9969.75,2024-12-01,t4,Daniel Metters,East


In [78]:
df_sum = transactions_with_territory.loc[
    lambda x: x["transaction_date"].dt.date.ge(date(2026, 1, 1)), 
    ["transaction_month", "territory_name", "amount_eur"]
].groupby(
    ["transaction_month", "territory_name"],
    as_index=False
).agg("sum").sort_values(
    ["territory_name", "transaction_month"]
)

df_sum["cumsum"] = df_sum.sort_values(
    ["territory_name", "transaction_month"]
).groupby(
    ["territory_name"])["amount_eur"].agg("cumsum")


df_sum

,transaction_month,territory_name,amount_eur,cumsum
0,2026-01-01,East,9594.30,9594.30
4,2026-02-01,East,7504.95,17099.25
11,2026-04-01,East,29473.51,46572.76
15,2026-05-01,East,8200.97,54773.73
19,2026-06-01,East,15792.29,70566.02
1,2026-01-01,North,52128.87,52128.87
5,2026-02-01,North,32742.59,84871.46
8,2026-03-01,North,69023.21,153894.67
12,2026-04-01,North,66325.53,220220.20
16,2026-05-01,North,36927.71,257147.91


In [48]:
df_sum.assign(cumulative_sum = lambda x: x["amount_eur"].cumsum())

,transaction_month,amount_eur,cumulative_sum
0,2026-01-01,145963.50,145963.50
1,2026-02-01,75080.64,221044.14
2,2026-03-01,174010.45,395054.59
3,2026-04-01,102827.27,497881.86
4,2026-05-01,119490.40,617372.26
5,2026-06-01,131196.11,748568.37
